# Reconstructing the pulse amplitude

_Author: Àfrica González Pedraza, Danaé Valdenaire_<br>
_Created: May 28th, 2026_<br>
_Last updated: June 25th, 2026 by Danaé Valdenaire_

---

This tutorial teaches how to reconstruct the pulse amplitudes employing:

- Standard Event Fit (SEV fit)
- Optimum Filter (OF)

While both the OF and the truncated SEV fit methods provide similar, reliable event
amplitude reconstruction up to the truncation limit (TL), they have distinct advantages:

- The SEV fit allows you to reconstruct the event amplitudes above the TL (which is impossible with the OF)
- The OF enhances the signal-to-noise ratio. Therefore, it
outperforms the SEV fit method when precisely reconstructing sub-keV/low-energy events, where the signal-to-noise ratio is low.

## First, you need (mock) data

In [ ]:
import os
import numpy as np
import cait as ai
import cait.versatile as vai

Before starting the work, we need to create our [`DataHandler`](cait.DataHandler) that contains our data. If you are not familiar with `DataHandler`, I really advice you to have a look at the `triggering tutorial`. All the steps from the creation to the DataHandler to the triggering of the stream data are well described there. 

In [ ]:
# REMOVE CELL BEFORE UPLOADING

fdirh5 = "tutorial_output"
hdf5_name = "my_first_trigger"
os.makedirs(fdirh5, exist_ok=True)

record_length = 2**14

trigger_config = {
    "trigger_channels": ["Ch0"],
    "passive_channels": ["Ch1"],
    "testpulse_channels": ["TP0", "TP1"],
    "controlpulses_above": [9., 9.],
    "f_noise": 1000,
    "copy_events": True,
}
n_channels = len(trigger_config["trigger_channels"]) + len(trigger_config["passive_channels"])

stream = vai.MockStream(seed=137, rate_Hz=2)

# Set the path to the desired HDF5 file
dh = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)

dh.set_filepath(path_h5="tutorial_output", fname="my_first_trigger", appendix=False)

## Building SEV, NPS, OF

This section is detailled in the tutorial called `tutorial_sev`.

### SEV - events

In [ ]:
quality_cuts = ai.cuts.LogicalCut()
quality_cuts.add_condition((dh["events/onset", 0]>-0.8)*(dh["events/onset", 0]<-0.4)); print(quality_cuts.counts())
quality_cuts.add_condition((dh["events/decay_time", 0]<40)); print(quality_cuts.counts())
quality_cuts.add_condition((dh["events/pulse_height", 0]>0.1)); print(quality_cuts.counts())

In [ ]:
dh.apply_logical_cut(cut_flag=quality_cuts.get_flag(),                                                             
                     naming='quality_cuts',
                     channel=0,
                     type='events',
                     delete_old=True)

In [ ]:
quality_cuts = dh["events/quality_cuts",0]

In [ ]:
event_iterator = dh.get_event_iterator(group="events", channel=0, flag=quality_cuts)
sev = vai.SEV(event_iterator)
fitpar, _, rms = vai.apply(vai.TemplateFit(sev, bl_poly_order=1), event_iterator)
sev = vai.SEV(event_iterator[:, rms<0.005])
dh_SEV = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)
dh_SEV.set_filepath(path_h5="tutorial_output", fname="my_SEV_OF_NPS", appendix=False)
dh_SEV.init_empty()
sev.to_dh(dh_SEV,"events","stdevent", overwrite_existing=True)

### SEV - testpulses

In [ ]:
quality_cuts_TP = ai.cuts.LogicalCut()
quality_cuts_TP.add_condition((dh["testpulses/testpulseamplitude", 0]==1)); print(quality_cuts_TP.counts())

dh.apply_logical_cut(cut_flag=quality_cuts_TP.get_flag(),                                                             
                     naming='quality_cuts_TP',
                     channel=0,
                     type='testpulses',
                     delete_old=True)

In [ ]:
quality_cuts_TP = dh["testpulses/quality_cuts_TP",0]
event_iterator = dh.get_event_iterator(group="testpulses", channel=0, flag=quality_cuts_TP)
sev = vai.SEV(event_iterator)
fitpar, _, rms = vai.apply(vai.TemplateFit(sev, bl_poly_order=1), event_iterator)
sev = vai.SEV(event_iterator[:, rms<0.00505])

In [ ]:
dh_SEV = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)
dh_SEV.set_filepath(path_h5="tutorial_output", fname="my_SEV_OF_NPS", appendix=False)
dh_SEV.init_empty()
sev.to_dh(dh_SEV,"testpulses","stdevent", overwrite_existing=True)

### NPS

In [ ]:
# performing quality cuts again
noise_cuts = ai.cuts.LogicalCut()
noise_cuts.add_condition(abs(dh["noise/pulse_height", 0])<0.05); print("Surviving after cut 1:", noise_cuts.counts())
noise_cuts.add_condition(dh["noise/variance", 0]<0.05); print("Surviving after cut 2:", noise_cuts.counts())

In [ ]:
noise_cuts=dh.get("noise","noise_cuts")[0] # Load the SEV cut from the datahandler, if you have saved it there. Alternatively, define it as in the beginning of this notebook.
noise_events = dh.get_event_iterator("noise")[0, noise_cuts].with_processing(vai.RemoveBaseline())
_, fit_rms = vai.apply(vai.FitBaseline(model=3, where=1.0), noise_events.with_batchsize(10))

In [ ]:
mask_noise = dh["noise/noise_cuts", 0] == True
mask_rms = fit_rms < 0.006   
full_mask = np.zeros_like(mask_noise, dtype=bool)
full_mask[mask_noise] = mask_rms
noise_cuts = ai.cuts.LogicalCut()
noise_cuts.add_condition(full_mask)
dh.apply_logical_cut(cut_flag=noise_cuts.get_flag(),                                   
                     naming='noise_cuts',
                     channel=0,
                     type='noise',
                     delete_old=True)

In [ ]:
noise_quality_cuts = (dh.get("noise", "noise_cuts")[0,:] == True)
noise_traces_ch0 = dh.get_event_iterator("noise", channel=0, flag=noise_quality_cuts)
nps0 = vai.NPS(noise_traces_ch0) 
vai.NPS([nps0],dt_us=dh.dt_us).to_dh(dh_SEV, overwrite_existing=True)  #Store in the dh

### OF

In [ ]:
# REMOVE CELL BEFORE UPLOADING

dh_SEV = ai.DataHandler(record_length=record_length, 
                    nmbr_channels=n_channels, 
                    sample_frequency=stream.sample_frequency)
dh_SEV.set_filepath(path_h5="tutorial_output", fname="my_SEV_OF_NPS", appendix=False)

In [ ]:
sev = vai.SEV().from_dh(dh_SEV, "stdevent") 
nps = vai.NPS().from_dh(dh_SEV, "nps")      
#vai.OF(sev, nps).to_dh(dh_SEV)              
of = vai.OF().from_dh(dh_SEV, "optimumfilter")  

## Standard event fit

In the following, the **truncated SEV fit** method is used to reconstruct the energy of the pulses. It consists of fitting the SEV (see `tutorial_sev.ipynb`) to each event, where the resulting amplitude is proportional to the energy deposition. When the detector exits the linear range of the phase transition, the pulse saturates. In this case, only samples below a **truncation limit (TL)** are included in the fit.

**The procedure is as follows:**

1. **Approximate the TL**: Fit the SEV without truncation and plot the fit RMS versus the SEV amplitude. The TL corresponds to the amplitude at which the RMS starts increasing, marking the onset of saturation.

2. **Apply the truncated SEV fit** to all events surviving the stability cuts.

3. **Quality cuts**: Remove events with a high RMS or an inconsistent amplitude. Ideally, fit the RMS distribution with a Gaussian or Rayleigh function and cut within a chosen number of standard deviations around the mean.

Events surviving these cuts are used for the energy spectrum reconstruction.

### Template fit

We start by fitting the SEV to the entire event spectrum without truncation to spot where the detector enters the non-linear regime. The truncation limit is our benchmark to distinguish between the linear and the non-linear regime.

In [ ]:
sev = dh_SEV.get("stdevent", "event")

dh.apply_template_fit(group="events", 
                      sev=sev, 
                      bl_poly_order=3,
                      only_channels=0,
                      #preview=True, #First preview that the fit is working as expected, then set to False to run the full fit.
                     )

# Output: templatefit_pars, templatefit_rms, templatefit_shift

Next, visualize the SEV fit amplitude vs. the RMS. While scatter plots are faster to render, they contain less information; therefore, density plots are recommended. It is important to zoom in on the different regions. Finally, select the TL below the amplitude where the RMS starts to increase.

#### Plotting with `versatile` 

In [ ]:
fitted_amplitude = dh["events/templatefit_pars",0,:,0]
fit_rms = dh["events/templatefit_rms",0]

In [ ]:
vai.Scatter(fitted_ampltiude[fit_rms>0], 
            fit_rms[fit_rms>0],
            xlabel='SEV fit amplitude (V)',
            ylabel='SEV fit rms (V)')

#### Plotting with Viztool

In [ ]:
datasets = {
    'Time (h)': ['hours', None, None],
    'Pulse Height Phonon (V)': ['pulse_height', 0, None],
    "Amplitude (V) ": ["templatefit_pars", 0, 0, None],
    "Shift ": ["templatefit_shift", 0, None],
    "RMS ": ["templatefit_rms", 0,  None],
}

In [ ]:
viz = ai.VizTool(datahandler = dh,
              group='events',
              datasets=datasets,
              bins = 100)
viz.show()

### Template fit with truncation limit

Now, you can run the template fit by adding the truncation limit. The fit will reconstruct all the saturated pulses so you can access the energies above the saturation. Don't forget to verify if the fit is done properly using the `Preview` functionnality. 

```{tips}
In the parameter ``tag``, always write '_trunc' followed by the TL, for example ``_trunc1p2`` to symbolize a TL of 1.2. This is helpful for the following reasons:
- Remember the chosen TL (it is not saved automatically!)
- Probe how different TLs or onsets affect your results (optional).
- Distinguish between the truncated and not-truncated SEV fit parameters.
```

In [ ]:
truncation_limit = 0.3

In [ ]:
dh.apply_template_fit(group="events", 
                      sev=sev, 
                      bl_poly_order=3,
                      truncation_limit=truncation_limit,
                      only_channels=0,
                      tag='_TL_3e-2',
                      #preview=True  #First preview that the fit is working as expected, then set to False to run the full fit.
                     )

```{note}
If you wish to re-do the fit, you must delete the previous fit before re-applying the fit with the new parameters. You can do this by running:
`dh.drop("events", "templatefit_pars_TL_xxx")
```

### Template fit with `versatile`

Attention, the fit is done on testpulses in this part.

If you want to preview the fit, do:

In [68]:
sev = vai.SEV().from_dh(dh_SEV, "testpulses", "stdevent")

In [69]:
vai.Preview(
    dh.get_event_iterator("testpulses", channel=0).with_processing(vai.RemoveBaseline()), 
    vai.TemplateFit(sev=sev,                                                   
                    bl_poly_order=3, 
                    truncation_limit=truncation_limit,
                   )
)

Then, use `vai.apply` to run the fit.

In [ ]:
event_iterator = dh.get_event_iterator("testpulses", channel=0).with_processing(vai.RemoveBaseline())
fitpar, _, rms = vai.apply(vai.TemplateFit(sev=sev, bl_poly_order=1, truncation_limit=truncation_limit), event_iterator)

In [ ]:
dh.set(group="testpulses", sev_fit_amplitude=fitpar.T[0], overwrite_existing=True)
dh.set(group="testpulses", sev_fit_rms=rms, overwrite_existing=True)

```{tip}
If you have multiple channels, use the syntax `dh.set(sev_fit_amp=[fitpar0.T[0], fitpar1[0],...])`
```

## Reconstructing the amplitude with an optimum filter (OF)

The energy reconstruction below TL can be best performed by filtering the recorded traces with
the optimum filter (OF). The OF is a frequency filter that considers both the noise power spectrum (NPS) and the
standard event, enhancing the characteristic frequencies present in the signal while suppressing those
dominant in the noise. While the template fit is powerful to access the higher energies, the OF can reconstruct efficiently the low energies. In a sense, those two methods are complementary.

In [ ]:
of = vai.OF().from_dh(dh_SEV, group="optimumfilter")
sev = vai.SEV().from_dh(dh_SEV, group="stdevent")

In [ ]:
dh.apply_ofilter(
    "events",
    of=of,
    sev=sev,
    only_channels=0,
    #preview=True, # set to False to fit
)

# Output: of_ph, of_max_val, of_eval_pos, of_max_pos, of_rms, of_peak_rms

We can also apply the OF on testpulses but we have to be careful to build an OF with a **testpulse template** and not an event template.

In [63]:
sev_tp = vai.SEV.from_dh(dh_SEV, "testpulses", "stdevent")
nps = vai.NPS().from_dh(dh_SEV)
of_tp = vai.OF(sev_tp, nps)
of_tp.to_dh(dh_SEV, group="optimumfilter_tp", overwrite_existing=True)

Successfully written optimumfilter_real with shape (1, 8193) and dtype 'float32' to group optimumfilter_tp.
Successfully written optimumfilter_imag with shape (1, 8193) and dtype 'float32' to group optimumfilter_tp.


In [65]:
of.show()

    'data': [{'line': {'width': 3},
              'mode': 'lines',
             …

In [66]:
dh.apply_ofilter("testpulses",
                 of=of_tp,
                 sev=sev_tp,
                 only_channels=0,
                 preview=True, # set to False to fit
                )

# Output: of_ph, of_max_val, of_eval_pos, of_max_pos, of_rms, of_peak_rms

In [67]:
dh.apply_template_fit("testpulses",
                     sev=sev_tp,
                     bl_poly_order=1,
                     only_channels=0,
                     truncation_limit=0.3,
                     preview=True, # set to False to fit
                     )

# Output: of_ph, of_max_val, of_eval_pos, of_max_pos, of_rms, of_peak_rms

### With `Viztool`

Inspect the RMS vs. OF amplitude plot alongside the other parameters. Here, the RMS is computed between the filtered SEV and the filtered event traces, while the peak RMS is calculated similarly but restricted to data points around the peak position.
When you are satisfied, apply cuts on the OF amplitude (using the TL as an upper bound) and on the RMS or peak RMS to keep only well-reconstructed events.

In [ ]:
datasets = {
    'Time (h)': ['hours', None, None],
    'Pulse Height Phonon (V)': ['pulse_height', 0, None],
    "OF Amplitude (V) ": ["of_ph", 0, None],
    "OF Max Val ": ["of_max_val", 0, None],
    "OF Eval Pos ": ["of_eval_pos", 0, None],
    "OF Max Pos ": ["of_max_pos", 0, None],
    "OF RMS ": ["of_rms", 0, None],
    "OF Peak RMS ": ["of_peak_rms", 0, None],
}

In [ ]:
viz = ai.VizTool(datahandler = dh,
              group = 'events',
              datasets = datasets,
              bins = 100)

viz.show()

## Advanced